# Practice: Auto Loader schema evolution and re-sent files
This lab copies one real Azure cost export file into a scratch folder and loads it with Auto Loader. It then simulates Azure re-sending the same month in a new run folder, with one extra column (`x_DiscountCode`). You will see two things:

* **Schema evolution:** in the default `addNewColumns` mode, the run that finds the new column stops with `UnknownFieldException` and saves a new schema version. The next run adds the column and loads the file.
* **Duplicates from re-sent files:** Auto Loader tracks files, not rows. The re-sent file is a new file, so its 623 rows are loaded again and the table ends up with every row twice.

Only scratch locations are used: `finops.raw.practice`, `/Volumes/finops/raw/practice/resend_demo`, `/Volumes/finops/raw/checkpoints/resend_demo` and `finops.bronze.resend_demo`. The real export folder is only read, and existing bronze tables are not touched.

**Run the cells one at a time, top to bottom.** Step 6 fails on purpose. Databricks marks a cell as failed when a stream inside it stops, even though the code catches the error, so *Run all* would stop at step 6.

In [0]:
from pyspark.sql import functions as F

# Real export (only read from) and the scratch locations this lab uses
REAL_EXPORT  = "/Volumes/finops/raw/real_exports/exports/george-test-focus-cost/"
PRACTICE_DIR = "/Volumes/finops/raw/practice/resend_demo"
CHECKPOINT   = "/Volumes/finops/raw/checkpoints/resend_demo"
TARGET_TABLE = "finops.bronze.resend_demo"

# Scratch volume for practice files (no change if it already exists)
spark.sql("CREATE VOLUME IF NOT EXISTS finops.raw.practice COMMENT 'Scratch files for practice labs'")

In [0]:
# Step 1 - Delete the practice folder, checkpoint and table so the lab starts fresh every time
dbutils.fs.rm(PRACTICE_DIR, recurse=True)
dbutils.fs.rm(CHECKPOINT, recurse=True)
spark.sql(f"DROP TABLE IF EXISTS {TARGET_TABLE}")
print("Scratch folder, checkpoint and table removed")

In [0]:
# Step 2 - Find the billing period folder and its latest run folder in the real export (read only)
period_dir = [d for d in dbutils.fs.ls(REAL_EXPORT) if d.isDir()][0]
run_dir    = sorted((d for d in dbutils.fs.ls(period_dir.path) if d.isDir()), key=lambda d: d.modificationTime)[-1]
PERIOD     = period_dir.name.rstrip("/")

# Copy the data file byte for byte, so every value stays exactly as Azure wrote it
RUN1_FILE = f"{PRACTICE_DIR}/{PERIOD}/run1/part_0_0001.csv.gz"
dbutils.fs.cp(run_dir.path + "part_0_0001.csv.gz", RUN1_FILE)
print("copied to", RUN1_FILE)

In [0]:
# Step 3 - One Auto Loader run: load any new files in PRACTICE_DIR into TARGET_TABLE, then stop
def run_auto_loader():
    stream = (spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("pathGlobFilter", "part_*.csv.gz")          # data files only
        .option("cloudFiles.schemaLocation", CHECKPOINT)    # schema versions are saved in _schemas/
        # Schema evolution mode left at the default (addNewColumns): stop on a new column, save it, continue next run
        .load(PRACTICE_DIR)
        .select("*",
                F.col("_metadata.file_path").alias("source_file"),
                F.current_timestamp().alias("ingested_at")))

    query = (stream.writeStream
        .option("checkpointLocation", CHECKPOINT)            # remembers which files were already loaded
        .option("mergeSchema", "true")                       # lets the Delta table accept the new column
        .trigger(availableNow=True)                          # the only trigger serverless allows
        .toTable(TARGET_TABLE))
    query.awaitTermination()

    # Use the .numInputRows attribute: on serverless, p["numInputRows"] always returns None.
    # A progress entry with nothing to read can still report None, so count that as 0.
    print("rows read this run:", sum(p.numInputRows or 0 for p in query.recentProgress))

In [0]:
# Step 4 - First load: expect 623 rows from run1
run_auto_loader()

In [0]:
# Step 5 - Simulate Azure re-sending the same month: same rows plus x_DiscountCode, in a run2 folder
RUN2_FILE = f"{PRACTICE_DIR}/{PERIOD}/run2/part_0_0001.csv.gz"
STAGING   = f"{PRACTICE_DIR}/_staging_run2"

# Read without type inference, so every column stays a string; quotes are doubled ("") as in Azure's files
run2 = (spark.read
    .option("header", "true")
    .option("escape", '"')
    .csv(RUN1_FILE)
    .withColumn("x_DiscountCode", F.lit("PROMO10")))

# Spark writes a folder of part files: write a single gzip file, then move it to Azure's file name
(run2.coalesce(1).write.mode("overwrite")
    .option("header", "true")
    .option("escape", '"')
    .option("compression", "gzip")
    .csv(STAGING))
part_file = [f.path for f in dbutils.fs.ls(STAGING) if f.name.endswith(".csv.gz")][0]
dbutils.fs.mv(part_file, RUN2_FILE)
dbutils.fs.rm(STAGING, recurse=True)

print("wrote", RUN2_FILE, "with", run2.count(), "rows and", len(run2.columns), "columns")

The next cell **fails on purpose**. Auto Loader finds `x_DiscountCode`, which is not in the saved schema. It saves schema version 1 and stops the stream with `UnknownFieldException`. Nothing from run2 is loaded yet. The cell catches the error and prints its first line, but the notebook still shows the cell as failed with the full stream error. That is expected; continue with step 7.

In [0]:
from pyspark.errors import StreamingQueryException

# Step 6 - Expected to fail: the stream stops when it sees the new column
# Only the streaming error is caught, so any other problem (e.g. a skipped cell) still shows up
try:
    run_auto_loader()
except StreamingQueryException as e:
    print("Stream stopped:", str(e).strip().splitlines()[0])

In [0]:
# Step 7 - Run again: the new schema includes x_DiscountCode, so run2 loads (expect 623 more rows)
run_auto_loader()

In [0]:
# Step 8 - Each schema change is a new numbered file: 0 = first schema, 1 = schema with x_DiscountCode
display(dbutils.fs.ls(f"{CHECKPOINT}/_schemas"))

In [0]:
%sql
-- Step 9 - Rows per run: run2 repeats run1, so the table now holds every cost row twice.
-- x_DiscountCode is NULL for run1 rows (the column didn't exist yet) and filled for run2 rows.
SELECT
  regexp_extract(source_file, '/(run[0-9]+)/', 1) AS run,
  count(*)                                         AS rows,
  count(x_DiscountCode)                            AS rows_with_discount_code
FROM finops.bronze.resend_demo
GROUP BY ALL
ORDER BY run;